# exp005: EfficientNet V2-S SED Training

exp002 ベースで backbone を EfficientNet V2-S に変更。
疑似ラベル生成用の Teacher モデルを学習する。

**実行環境**: Kaggle Notebook or Google Colab (GPU)

**変更点 (vs exp002)**:
- `tf_efficientnet_b0_ns` -> `tf_efficientnetv2_s`
- batch_size: 32 -> 16 (V2-Sはパラメータが多い)
- epochs: 20 -> 30

In [ ]:
!pip install -q timm torchaudio scikit-learn

In [ ]:
import os, pathlib, glob

# ── Path setup ────────────────────────────────────────────
_s = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
COMP_DIR        = os.path.dirname(_s[0]) if _s else '/kaggle/input/birdclef-2026'
TRAIN_CSV       = f'{COMP_DIR}/train.csv'
TAXONOMY_CSV    = f'{COMP_DIR}/taxonomy.csv'
SAMPLE_SUB_CSV  = f'{COMP_DIR}/sample_submission.csv'
TRAIN_AUDIO_DIR = f'{COMP_DIR}/train_audio'

WEIGHT_DIR = '/kaggle/working/weights'
LOG_DIR    = '/kaggle/working/logs'
os.makedirs(WEIGHT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

for label, path in [('TRAIN_CSV', TRAIN_CSV), ('SAMPLE_SUB_CSV', SAMPLE_SUB_CSV), ('TRAIN_AUDIO_DIR', TRAIN_AUDIO_DIR)]:
    print(f'  {"OK" if pathlib.Path(path).exists() else "NG"} {label}: {path}')

In [ ]:
CFG = dict(
    # Audio
    sample_rate      = 32000,
    n_samples        = 32000 * 5,    # 5s (match prediction segment)
    n_mels           = 128,
    n_fft            = 1024,
    hop_length       = 320,
    fmin             = 20,
    fmax             = 16000,
    # Training
    seed             = 42,
    n_folds          = 5,
    train_fold       = 0,
    epochs           = 30,
    batch_size       = 32,           # increased (5s input uses less memory)
    num_workers      = 2,
    lr               = 1e-3,
    weight_decay     = 1e-4,
    warmup_epochs    = 1,
    use_amp          = True,
    label_smoothing  = 0.05,
    early_stopping   = 5,            # stop if no improvement for N epochs
    # Model
    model_name       = 'tf_efficientnetv2_s',
    num_classes      = 234,
    # SpecAugment
    spec_aug         = True,
    freq_mask_param  = 27,
    time_mask_param  = 100,
    # Gaussian Noise
    gaussian_noise   = True,
    noise_std        = 0.005,
    noise_prob       = 0.5,
)

In [ ]:
import ast, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import timm
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}, GPUs: {torch.cuda.device_count()}')

In [ ]:
# ── Utilities ─────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0.0
    def update(self, val, n=1):
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def compute_roc_auc(targets, preds, labels):
    aucs = []
    for i in range(len(labels)):
        if targets[:, i].sum() == 0:
            continue
        try:
            aucs.append(roc_auc_score(targets[:, i], preds[:, i]))
        except Exception:
            pass
    return float(np.mean(aucs)) if aucs else 0.0

set_seed(CFG['seed'])

In [ ]:
# ── Mel Transform & SpecAugment (GPU) ─────────────────────
mel_transform = nn.Sequential(
    T.MelSpectrogram(
        sample_rate=CFG['sample_rate'],
        n_fft=CFG['n_fft'],
        hop_length=CFG['hop_length'],
        n_mels=CFG['n_mels'],
        f_min=CFG['fmin'],
        f_max=CFG['fmax'],
    ),
    T.AmplitudeToDB(top_db=80),
).to(DEVICE)

freq_masking = T.FrequencyMasking(freq_mask_param=CFG['freq_mask_param']).to(DEVICE)
time_masking = T.TimeMasking(time_mask_param=CFG['time_mask_param']).to(DEVICE)

def waveform_to_spec(waveforms, training=False):
    with torch.no_grad():
        specs = mel_transform(waveforms)
    specs = specs - specs.amin(dim=(-2, -1), keepdim=True)
    specs = specs / (specs.amax(dim=(-2, -1), keepdim=True) + 1e-8)
    if training and CFG['spec_aug']:
        specs = freq_masking(specs)
        specs = time_masking(specs)
    return specs.unsqueeze(1)

In [ ]:
# ── Dataset ───────────────────────────────────────────────
class BirdCLEFDataset(Dataset):
    def __init__(self, df, label2idx, mode='train'):
        self.df = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.mode = mode
        self.n = CFG['n_samples']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio = self._load_audio(row['filename'])
        primary_idx = self.label2idx.get(row['primary_label'], 0)
        return audio, primary_idx

    def _load_audio(self, filename):
        path = f'{TRAIN_AUDIO_DIR}/{filename}'
        try:
            waveform, sr = torchaudio.load(path)
        except Exception:
            return torch.zeros(self.n)
        if sr != CFG['sample_rate']:
            waveform = torchaudio.functional.resample(waveform, sr, CFG['sample_rate'])
        audio = waveform.mean(dim=0)
        if len(audio) >= self.n:
            start = random.randint(0, len(audio) - self.n) if self.mode == 'train' else (len(audio) - self.n) // 2
            audio = audio[start:start + self.n]
        else:
            audio = F.pad(audio, (0, self.n - len(audio)))
        if self.mode == 'train' and CFG['gaussian_noise'] and random.random() < CFG['noise_prob']:
            audio = audio + torch.randn_like(audio) * CFG['noise_std']
        return audio

In [ ]:
# ── Model ─────────────────────────────────────────────────
class AttBlockV2(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.att = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, num_classes, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        nn.init.constant_(self.att.bias, 0)
        nn.init.constant_(self.cla.bias, 0)

    def forward(self, x):
        att = torch.softmax(torch.tanh(self.att(x)), dim=-1)
        cla = self.cla(x)
        return (att * cla).sum(dim=-1)

class BirdCLEFSED(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            CFG['model_name'],
            pretrained=pretrained,
            in_chans=1,
            num_classes=0,
            global_pool='',
        )
        in_features = self.backbone.num_features
        self.bn = nn.BatchNorm1d(in_features)
        self.dropout = nn.Dropout(p=0.3)
        self.att_block = AttBlockV2(in_features, CFG['num_classes'])

    def forward(self, x):
        feat = self.backbone.forward_features(x)
        feat = feat.mean(dim=2)
        feat = self.bn(feat)
        feat = self.dropout(feat)
        return self.att_block(feat)

# Quick test
_m = BirdCLEFSED(pretrained=False)
print(f'Backbone features: {_m.backbone.num_features}')
_p = sum(p.numel() for p in _m.parameters())
print(f'Total params: {_p/1e6:.1f}M')
del _m

In [ ]:
# ── Data Loading & Fold Split ─────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
sub_df = pd.read_csv(SAMPLE_SUB_CSV, nrows=0)
LABELS = [c for c in sub_df.columns if c != 'row_id']
LABEL2IDX = {l: i for i, l in enumerate(LABELS)}

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    train_df.loc[val_idx, 'fold'] = fold

FOLD = CFG['train_fold']
tr_df = train_df[train_df['fold'] != FOLD].reset_index(drop=True)
va_df = train_df[train_df['fold'] == FOLD].reset_index(drop=True)

print(f'Fold {FOLD}: train={len(tr_df)}, valid={len(va_df)}, classes={len(LABELS)}')

In [ ]:
# ── DataLoader with WeightedRandomSampler ─────────────────
counts = tr_df['primary_label'].value_counts()
total = len(tr_df)
sample_weights = tr_df['primary_label'].map(
    lambda x: (counts.get(x, 1) / total) ** (-0.5)
).values
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float64),
    num_samples=len(sample_weights),
    replacement=True,
)

tr_loader = DataLoader(
    BirdCLEFDataset(tr_df, LABEL2IDX, mode='train'),
    batch_size=CFG['batch_size'],
    sampler=sampler,
    num_workers=CFG['num_workers'],
    pin_memory=True,
    drop_last=True,
)
va_loader = DataLoader(
    BirdCLEFDataset(va_df, LABEL2IDX, mode='valid'),
    batch_size=CFG['batch_size'] * 2,
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=True,
)
print(f'Train batches: {len(tr_loader)}, Valid batches: {len(va_loader)}')

In [ ]:
# ── Training & Validation Loop ────────────────────────────
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    loss_meter = AverageMeter()
    for waveforms, targets in tqdm(loader, desc='  train', leave=False):
        waveforms = waveforms.to(DEVICE)
        targets = targets.to(DEVICE)
        optimizer.zero_grad()
        # mel spectrogram must run in float32 (AmplitudeToDB amin=1e-10 underflows in fp16)
        specs = waveform_to_spec(waveforms, training=True)
        with autocast(enabled=CFG['use_amp']):
            logits = model(specs)
            loss = F.cross_entropy(logits, targets, label_smoothing=CFG['label_smoothing'])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_meter.update(loss.item(), waveforms.size(0))
    return loss_meter.avg

@torch.no_grad()
def validate(model, loader):
    model.eval()
    loss_meter = AverageMeter()
    all_preds, all_targets = [], []
    for waveforms, targets in tqdm(loader, desc='  valid', leave=False):
        waveforms = waveforms.to(DEVICE)
        targets = targets.to(DEVICE)
        # mel spectrogram must run in float32
        specs = waveform_to_spec(waveforms, training=False)
        with autocast(enabled=CFG['use_amp']):
            logits = model(specs)
            loss = F.cross_entropy(logits, targets, label_smoothing=CFG['label_smoothing'])
        loss_meter.update(loss.item(), waveforms.size(0))
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(probs)
        one_hot = np.zeros((len(targets), len(LABELS)), dtype=np.float32)
        for i, t in enumerate(targets.cpu().numpy()):
            one_hot[i, t] = 1.0
        all_targets.append(one_hot)
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    auc = compute_roc_auc(all_targets, all_preds, LABELS)
    return loss_meter.avg, auc

In [ ]:
# ── Training ──────────────────────────────────────────────
model = BirdCLEFSED(pretrained=True).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)
scaler = GradScaler(enabled=CFG['use_amp'])

WEIGHT_PATH = f'{WEIGHT_DIR}/best_fold{FOLD}.pth'
CKPT_PATH = f'{WEIGHT_DIR}/checkpoint_fold{FOLD}.pth'
best_auc, best_epoch = 0.0, 0
start_epoch = 1
log_rows = []

# Resume from checkpoint
if os.path.exists(CKPT_PATH):
    print(f'Resuming from checkpoint: {CKPT_PATH}')
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_auc = ckpt['best_auc']
    log_rows = ckpt.get('log_rows', [])
    print(f'  Resumed from epoch {ckpt["epoch"]} | best AUC: {best_auc:.4f}')
else:
    print('Starting from scratch.')

print('=' * 60)
print(f'exp005 | Fold {FOLD} | epoch {start_epoch}~{CFG["epochs"]} | {DEVICE}')
print(f'Model: SED + AttBlockV2 + {CFG["model_name"]}')
print(f'Input: {CFG["n_samples"]/CFG["sample_rate"]:.0f}s | AMP: {CFG["use_amp"]}')
print(f'Early stopping: {CFG["early_stopping"]} epochs')
print('=' * 60)

for epoch in range(start_epoch, CFG['epochs'] + 1):
    if epoch <= CFG['warmup_epochs']:
        for pg in optimizer.param_groups:
            pg['lr'] = CFG['lr'] * epoch / CFG['warmup_epochs']

    tr_loss = train_one_epoch(model, tr_loader, optimizer, scaler)

    if epoch > CFG['warmup_epochs']:
        scheduler.step()

    va_loss, va_auc = validate(model, va_loader)
    lr = optimizer.param_groups[0]['lr']
    log_rows.append(dict(epoch=epoch, lr=lr, tr_loss=tr_loss, va_loss=va_loss, va_auc=va_auc))

    is_best = va_auc > best_auc
    print(f'Epoch {epoch:03d}/{CFG["epochs"]}'
          f' | LR={lr:.2e} | Train={tr_loss:.4f} | Valid={va_loss:.4f} | AUC={va_auc:.4f}'
          f'{" <- best" if is_best else ""}')

    if is_best:
        best_auc, best_epoch = va_auc, epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_auc': best_auc,
            'labels': LABELS,
            'cfg': CFG,
        }, WEIGHT_PATH)

    # Early stopping
    if epoch - best_epoch >= CFG['early_stopping']:
        print(f'Early stopping: no improvement for {CFG["early_stopping"]} epochs')
        break

    if epoch % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_auc': best_auc,
            'labels': LABELS,
            'cfg': CFG,
            'log_rows': log_rows,
        }, CKPT_PATH)
        print(f'  Checkpoint saved (epoch {epoch})')

    pd.DataFrame(log_rows).to_csv(f'{LOG_DIR}/train_log_fold{FOLD}.csv', index=False)

print('=' * 60)
print(f'Best AUC: {best_auc:.4f} @ Epoch {best_epoch}')
print('=' * 60)

In [ ]:
# ── Training Curve ────────────────────────────────────────
import matplotlib.pyplot as plt

log_df = pd.DataFrame(log_rows)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ['tr_loss', 'va_loss', 'va_auc'], ['Train Loss', 'Valid Loss', 'Valid AUC']):
    ax.plot(log_df['epoch'], log_df[col])
    ax.set_title(title)
    ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/train_curve_fold{FOLD}.png', dpi=100)
plt.show()
print(f'Best model: {WEIGHT_PATH}')

In [ ]:
# ── Next Steps ────────────────────────────────────────────
print('Training complete!')
print(f'Best model: {WEIGHT_PATH}')
print()
print('Next steps:')
print('1. Save Version -> Save & Run All')
print('2. Output -> weights/best_fold0.pth -> New Dataset: birdclef2026-exp005-weights')
print('3. Run pseudo_label.ipynb with this dataset as input')